# Version 5 — Pinecone + SQL Validation + PostgreSQL + Tavily Web Search

Version 5 is an extension of Version 4. It keeps the existing Pinecone retrieval tools, SQL validation, and PostgreSQL execution, and adds Tavily web search with domain filtering.

```text
                         User Question
                               |
                               v
                     AI Data Analyst Agent
                               |
          +--------------------+--------------------+
          |                    |                    |
          v                    v                    v
 retrieve_schema_metadata  retrieve_similar_sql  retrieve_business
          |                    |                 _context
          +--------------------+--------------------+
                               |
                      Need external facts?
                         /           \
                       No             Yes
                       |               |
                       |               v
                       |        web_search(query,
                       |          include_domains,
                       |          exclude_domains)
                       |               |
                       +-------+-------+
                               |
                               v
                          Generate SQL
                               |
                               v
                       validate_query
                               |
                     +---------+---------+
                     |                   |
                  Failed               Passed
                     |                   |
                     v                   v
               Correct SQL       run_query
               and revalidate             |
                                          v
                                  PostgreSQL Results
                                          |
                                          v
                                   Final Answer
```

The agent should call only the tools needed for the question. Web search is optional and does not replace SQL for internal business data.


# Version 3 — Existing Pinecone Knowledge Base + PostgreSQL

This version connects to the Pinecone index shown in your screenshots (`ecom-kb`) and searches its three namespaces: `nlq-schema`, `nlq-examples`, and `common`. It does not rebuild or upload the knowledge base.

**Before running:** set `PINECONE_API_KEY`, `OPENAI_API_KEY`, `BEDROCK_CHAT_MODEL_ID`, and PostgreSQL environment variables. The query embedding model and dimension must match the model used when vectors were upserted to Pinecone.


## End-to-end agent workflow — Version 5

```text
                         User Question
                               |
                               v
                       AI Data Analyst Agent
                               |
                +--------------+--------------+
                |              |              |
                v              v              v
          Internal data?   External facts?   Both?
                |              |              |
                v              v              v
        Pinecone lookup    web_search()    Pinecone lookup
        when needed             |          + web_search()
                |               |              |
                v               |              v
        Generate SQL            |       Generate SQL if needed
                |               |              |
                v               |              v
        validate_query()        |       validate_query() if SQL
                |               |              |
          Pass / Fail            |          Pass / Fail
                |               |              |
                v               |              v
        run_query() if valid     |       run_query() if valid
                |               |              |
                +---------------+--------------+
                                |
                                v
               Synthesize only the evidence collected
                                |
                                v
           Answer with relevant sources and caveats
```

The agent selects only the tools needed. Web search does not replace PostgreSQL for internal data, and web-only questions do not require SQL.

# Version 3 — Add Semantic Schema Memory to the Data Analyst Agent

In Version 2, our agent had a **brain and hands**:
- **Brain:** the language model.
- **Hands:** `run_query(sql)`, which executes read-only SQL against PostgreSQL.

In Version 3, we add a **knowledge lookup tool** called `lookup_schema(term)`.

This tool searches a semantic memory containing schema knowledge: table descriptions, column definitions, synonyms, and join guidance. The agent can retrieve the relevant schema before writing SQL, instead of guessing column names.

**Scope:** This version adapts only the `lookup_schema` capability from `nlq.zip`. It does not add the other tools from that project.

## 1. The new architecture

```text
User question
     |
     v
Agent + model (brain)
     |
     +---- lookup_schema(term) ----> Semantic schema memory
     |                                  |
     |<------- relevant schema ----------+
     |
     +---- run_query(sql) ----------> PostgreSQL
     |                                  |
     |<------- query results ------------+
     |
     v
Final answer grounded in query results
```

**Why semantic memory?** Users may ask about “buyers,” while the schema calls them `customers`; or ask for “money kept,” while the business definition uses payment status. Embedding-based retrieval can find relevant schema descriptions even when the wording differs.

## 2. Install dependencies

We use:
- `langchain-openai` for the model and embeddings.
- `langchain` for the agent and tools.
- `pandas` and `openpyxl` to read the knowledge-base workbook.
- `numpy` for cosine similarity.
- `python-dotenv` for environment variables.
- `psycopg` for PostgreSQL.

In [ ]:
%pip install -q langchain langchain-aws langchain-openai pinecone requests python-dotenv "psycopg[binary]" tavily-python


## 3. Configure the model and database

Place `kb_feed_ecom.xlsx` in the same folder as this notebook, or update `KB_PATH`. Configure the API key and PostgreSQL connection in environment variables or a `.env` file.

Required environment variables:
- `OPENAI_API_KEY`
- `PGHOST`
- `PGDATABASE`
- `PGUSER`
- `PGPASSWORD`

Optional:
- `PGPORT` (defaults to `5432`)
- `PGSSLMODE` (set to the value required by your database provider)

In [ ]:
import os
import json
import psycopg
from psycopg.rows import dict_row
from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_aws import ChatBedrockConverse
from langchain_openai import OpenAIEmbeddings
from pinecone import Pinecone
from tavily import TavilyClient

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "ecom-kb")
EMBEDDING_MODEL = os.getenv("PINECONE_EMBEDDING_MODEL", "text-embedding-3-small")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
BEDROCK_CHAT_MODEL_ID = os.getenv("BEDROCK_CHAT_MODEL_ID", "")

if not PINECONE_API_KEY:
    raise ValueError("Set PINECONE_API_KEY.")
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY for query embeddings. It must match the embedding model used to build the Pinecone index.")
if not BEDROCK_CHAT_MODEL_ID:
    raise ValueError("Set BEDROCK_CHAT_MODEL_ID to a chat model enabled in your AWS account.")

required_pg_vars = ["PGHOST", "PGDATABASE", "PGUSER", "PGPASSWORD"]
missing_pg_vars = [name for name in required_pg_vars if not os.getenv(name)]
if missing_pg_vars:
    raise ValueError("Missing PostgreSQL environment variables: " + ", ".join(missing_pg_vars))

PGPORT = int(os.getenv("PGPORT", "5432"))
PGSSLMODE = os.getenv("PGSSLMODE")

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
pinecone = Pinecone(api_key=PINECONE_API_KEY)
index = pinecone.Index(PINECONE_INDEX_NAME)
model = ChatBedrockConverse(
    model=BEDROCK_CHAT_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0,
)


# Tavily is used only for external/current web information.
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY", "")
tavily_client = TavilyClient(api_key=TAVILY_API_KEY) if TAVILY_API_KEY else None


## 4. Build the semantic schema memory

We use the workbook's `nlq_table` and `nlq_column` sheets as the source of truth for schema knowledge.

Each active table or column becomes a searchable document. We embed those documents once, then use cosine similarity to retrieve the most relevant records for a lookup.

This is a small, in-notebook semantic index for teaching. It is not a persistent vector database. For production, the same retrieval pattern can be backed by Pinecone or another vector store.

## 5. Connect to the existing Pinecone knowledge base

Important: the workbook has a separate `nlq_examples` sheet. Schema lookup alone cannot retrieve worked SQL examples. We load those questions, paraphrases, and verified SQL statements into a second semantic index so the agent can find a similar example before drafting a query.


## 5. Embed the schema documents

The embedding model converts schema descriptions into vectors. The lookup tool embeds the user's search phrase and compares it with these vectors.

For a large knowledge base, store the vectors in a persistent vector index instead of recomputing them every time the notebook starts.

## 6. Add the `lookup_schema(term)` tool

This is the new **knowledge lookup tool**. It does not query PostgreSQL. It searches schema memory and returns relevant table and column definitions to the agent.

Like the schema lookup tool in `nlq.zip`, it:
- Searches for the schema concept requested by the agent.
- Returns relevant table/column descriptions.
- Records concepts already searched during the current user turn, so repeated rephrasings do not trigger repeated retrieval calls.

The repeat cache below is deliberately scoped to a single `ask_agent()` call for this notebook demonstration.

## 7. Keep the PostgreSQL tool: `run_query(sql)`

This remains the agent's **hands** for database execution. The new `lookup_schema` tool is its **semantic memory**. They have different responsibilities:
- `lookup_schema` retrieves knowledge about the schema.
- `run_query` executes SQL and returns database results.

## 8. Add `lookup_examples(question)`

`lookup_examples` searches the existing Pinecone examples index. `lookup_schema` answers **which tables and columns mean what**; `lookup_examples` answers **has a similar question already been solved, and what SQL pattern was used?**


## Pinecone hybrid retrieval: vector kNN + BM25

The supplied `nlq.zip` code uses Pinecone indexes, Bedrock Titan embeddings, and AWS SigV4 authentication. A query is sent to both the vector field (`embedding`) and text field (`text`), then the index returns matching records.

**Important:** the supplied `nlq.zip` implementation uses Pinecone, while your screenshots show Pinecone. This notebook follows the live Pinecone index shown in your screenshots.


## Add `validate_query(sql)` before execution

This tool performs a read-only safety and syntax pre-check before `run_query`. It rejects multiple statements and non-read-only SQL, checks PostgreSQL syntax with `EXPLAIN` without executing the query, and returns actionable errors. It is a guardrail, not a substitute for database permissions.


In [ ]:
@tool
def run_query(sql: str) -> str:
    """Execute one read-only SELECT query against the configured PostgreSQL database.

    Use only after consulting lookup_schema when table/column details are not
    already established. Returns at most 50 rows.
    """
    sql_clean = sql.strip()
    if not sql_clean:
        return "REFUSED: SQL query is empty."

    first_word = sql_clean.split(None, 1)[0].upper()
    if first_word not in {"SELECT", "WITH"}:
        return "REFUSED: Only read-only SELECT queries are allowed."

    connection_kwargs = {
        "host": os.environ["PGHOST"],
        "port": PGPORT,
        "dbname": os.environ["PGDATABASE"],
        "user": os.environ["PGUSER"],
        "password": os.environ["PGPASSWORD"],
        "connect_timeout": 10,
        "row_factory": dict_row,
    }
    if PGSSLMODE:
        connection_kwargs["sslmode"] = PGSSLMODE

    try:
        with psycopg.connect(**connection_kwargs) as conn:
            with conn.transaction():
                conn.execute("SET TRANSACTION READ ONLY")
                conn.execute("SET LOCAL statement_timeout = '15000ms'")
                cursor = conn.execute(sql_clean)
                rows = cursor.fetchmany(50)
                columns = [item.name for item in cursor.description] if cursor.description else []

        if not rows:
            return "Query succeeded. 0 rows returned."

        header = " | ".join(columns)
        body = "\n".join(
            " | ".join(str(row.get(column)) for column in columns)
            for row in rows
        )
        return f"{header}\n{body}\n(Returned at most 50 rows.)"

    except (psycopg.Error, ValueError) as exc:
        return f"SQL error: {exc}"

In [ ]:
import re


def _remove_sql_comments(sql: str) -> str:
    """Remove SQL comments before basic statement checks."""
    sql = re.sub(r"/\*.*?\*/", " ", sql, flags=re.S)
    sql = re.sub(r"--[^\n]*", " ", sql)
    return sql.strip()


def _validate_read_only_sql(sql: str) -> tuple[bool, str]:
    """Apply conservative checks before asking PostgreSQL to parse the query."""
    if not isinstance(sql, str) or not sql.strip():
        return False, "SQL must be a non-empty string."

    cleaned = _remove_sql_comments(sql).strip()
    if not cleaned:
        return False, "SQL is empty after removing comments."

    # Allow one trailing semicolon, but reject multiple statements.
    without_trailing = cleaned[:-1].rstrip() if cleaned.endswith(";") else cleaned
    if ";" in without_trailing:
        return False, "Only one SQL statement is allowed."

    # Only SELECT / WITH queries are accepted for this read-only analyst.
    if not re.match(r"(?is)^(SELECT|WITH)\b", without_trailing):
        return False, "Only read-only SELECT or WITH queries are allowed."

    # Reject common write/administrative operations even if hidden in a CTE.
    forbidden = re.compile(
        r"(?is)\b(INSERT|UPDATE|DELETE|MERGE|UPSERT|CREATE|ALTER|DROP|TRUNCATE|"
        r"GRANT|REVOKE|CALL|DO|COPY|VACUUM|ANALYZE|REFRESH|SET|RESET|"
        r"pg_sleep|lo_import|lo_export)\b"
    )
    if forbidden.search(without_trailing):
        return False, "The SQL contains a write or administrative operation and was rejected."

    return True, without_trailing


@tool
def validate_query(sql: str) -> str:
    """Validate a generated SQL query before execution.
    Checks that it is a single read-only SELECT/WITH statement and asks PostgreSQL
    EXPLAIN to parse and plan it without running the query. Call this before run_query.
    This tool does not execute the query or return business data."""
    is_valid, normalized_sql = _validate_read_only_sql(sql)
    if not is_valid:
        return f"VALIDATION FAILED: {normalized_sql}"

    conn_kwargs = {
        "host": os.environ["PGHOST"],
        "port": PGPORT,
        "dbname": os.environ["PGDATABASE"],
        "user": os.environ["PGUSER"],
        "password": os.environ["PGPASSWORD"],
        "connect_timeout": 5,
        "options": "-c default_transaction_read_only=on -c statement_timeout=10000",
    }
    if PGSSLMODE:
        conn_kwargs["sslmode"] = PGSSLMODE

    try:
        with psycopg.connect(**conn_kwargs, row_factory=dict_row) as conn:
            with conn.cursor() as cur:
                cur.execute("EXPLAIN " + normalized_sql)
                plan = cur.fetchall()

        plan_text = "\n".join(str(row.get("QUERY PLAN", row)) for row in plan[:8])
        return (
            "VALIDATION PASSED: PostgreSQL accepted the query for planning. "
            "The query has not been executed. You may now call execute_read_only_sql(sql).\n"
            f"Query plan preview:\n{plan_text[:3000]}"
        )
    except Exception as exc:
        # Return a short, actionable error without leaking connection details.
        message = str(exc).splitlines()[0][:500]
        return f"VALIDATION FAILED: PostgreSQL could not validate the query: {message}"


In [ ]:
def search_namespace(namespace: str, question: str, top_k: int = 8) -> list[dict]:
    """Embed a question and search the selected existing Pinecone namespace."""
    vector = embeddings.embed_query(question)
    response = index.query(
        vector=vector,
        namespace=namespace,
        top_k=top_k,
        include_metadata=True,
    )
    matches = response.get("matches", []) if isinstance(response, dict) else response.matches
    results = []
    for match in matches:
        if isinstance(match, dict):
            match_id = match.get("id", "unknown")
            score = match.get("score")
            metadata = match.get("metadata") or {}
        else:
            match_id = getattr(match, "id", "unknown")
            score = getattr(match, "score", None)
            metadata = getattr(match, "metadata", None) or {}
        results.append({"id": match_id, "score": score, "metadata": metadata})
    return results


def format_matches(matches: list[dict], max_chars: int = 12000) -> str:
    """Format Pinecone metadata while omitting vector payloads."""
    if not matches:
        return "No matching knowledge-base records were found."
    blocks = []
    for rank, match in enumerate(matches, start=1):
        lines = [
            f"RESULT {rank} | id={match['id']} | score={match['score']}"
        ]
        for key, value in match["metadata"].items():
            if isinstance(value, (dict, list)):
                value = json.dumps(value, ensure_ascii=False)
            lines.append(f"{key}: {value}")
        blocks.append("\n".join(lines))
    output = "\n\n---\n\n".join(blocks)
    return output[:max_chars] + ("\n...[truncated]" if len(output) > max_chars else "")


@tool
def lookup_schema(term: str) -> str:
    """Search the nlq-schema Pinecone namespace for tables and columns.
    Use it to confirm real table/column names, data types, descriptions, synonyms,
    and business meaning. Retrieves metadata only; does not execute SQL."""
    return format_matches(search_namespace("nlq-schema", term, top_k=8))


@tool
def lookup_examples(question: str) -> str:
    """Search the nlq-examples Pinecone namespace for similar approved NL-to-SQL examples.
    Use examples as patterns, then verify schema details using lookup_schema.
    Retrieves examples only; does not execute SQL."""
    return format_matches(search_namespace("nlq-examples", question, top_k=4), max_chars=10000)


@tool
def lookup_common(question: str) -> str:
    """Search the common Pinecone namespace for shared business definitions or guidance.
    Retrieves knowledge only; does not execute SQL."""
    return format_matches(search_namespace("common", question, top_k=4), max_chars=8000)


## 8. Update the system prompt

The agent now has five tools: schema lookup, SQL-example lookup, common-knowledge lookup, query validation, and database query execution. We explicitly explain when to use each one and tell the agent to consult semantic schema memory before guessing column names.

In [ ]:
SYSTEM_PROMPT = """
ROLE
You are an AI Data Analyst Agent for an e-commerce marketplace. Answer questions using
internal business data, internal knowledge-base context, and external public web
information when the question requires it.

AVAILABLE TOOLS
- lookup_schema(term): searches Pinecone namespace `nlq-schema`.
- lookup_examples(question): searches Pinecone namespace `nlq-examples`.
- lookup_common(question): searches Pinecone namespace `common`.
- validate_query(sql): checks a single read-only SELECT/WITH query and asks PostgreSQL
  EXPLAIN to validate its syntax/plan; it does not execute the business query.
- run_query(sql): executes a read-only SQL query against PostgreSQL and returns rows.
- web_search(query, include_domains=None, exclude_domains=None, max_results=5):
  searches public web sources using Tavily. Domain filters are optional.

ROUTING RULES
- Internal marketplace metrics, orders, customers, products, or transactions require
  PostgreSQL evidence. Use Pinecone retrieval when schema or business definitions are
  needed to formulate the query accurately.
- Current/external facts, competitor information, public reports, and industry
  benchmarks require web_search.
- Questions comparing internal performance with external benchmarks may require both
  PostgreSQL and web_search. Label the two evidence sources separately.
- For general knowledge questions that do not need internal data, do not generate SQL.
- Call only tools relevant to the question; do not call all tools by default.
- If the user specifies permitted domains, pass them to include_domains. If the user
  specifies domains to exclude, pass them to exclude_domains. Preserve these constraints.
- Do not present web results as facts about the user's internal business.

INTERNAL DATA / SQL WORKFLOW
- Identify the metric, population, filters, date range, and desired grain.
- Use lookup_schema, lookup_examples, and lookup_common when relevant.
- Treat retrieved examples as patterns, not guaranteed-correct SQL. Verify names,
  joins, filters, definitions, and aggregation.
- Never invent tables, columns, joins, value codes, or business definitions.
- Before executing SQL, call validate_query(sql).
- If validation fails, correct the SQL and validate it again.
- Call run_query(sql) only after validation succeeds.
- Inspect returned results before reporting them. Revalidate every revised query.
- Never claim validation or execution succeeded unless the corresponding tool confirms it.
- Validation is not a complete security boundary; enforce read-only access in the DB.

WEB EVIDENCE RULES
- Never invent search findings, source titles, URLs, dates, or citations.
- Use the source titles and URLs actually returned by web_search.
- Summarize source evidence accurately and distinguish facts from interpretation.
- If results are weak, conflicting, or insufficient, say so.
- Do not claim the web search is exhaustive.

GENERAL CONSTRAINTS
- Never invent numbers or query results.
- Consider table grain, join cardinality, NULLs, and double-counting.
- For percentages, state numerator and denominator.
- Do not expose private chain-of-thought; provide a concise explanation of method and evidence.

OUTPUT FORMAT
Choose only the sections appropriate to the question; do not force SQL sections onto a
web-only or general-knowledge answer.

ANSWER:
Give the answer with units where applicable, or state UNKNOWN if the evidence is
insufficient.

CALCULATION:
For percentages or derived metrics, explain the numerator, denominator, and method.
Omit if not applicable.

SOURCES:
For database answers, identify PostgreSQL and include the SQL actually executed if
successful. For web answers, list source titles and URLs returned by Tavily. For
combined answers, separate internal database evidence from external web evidence.
Omit sources that were not used.

CAVEATS:
State material limitations, ambiguity, time-period mismatches, data-quality concerns,
or source uncertainty. Omit if none are material.
"""

## 9. Create the agent with both tools

Version 2 had one tool. Version 3 now passes both tools to `create_agent()`.

## Version 5 addition — Tavily web search with domain filtering

Version 5 retains all Version 4 tools and adds `web_search()` for current or external information. Use `include_domains` to restrict sources and `exclude_domains` to omit domains. Set `TAVILY_API_KEY` in your environment or `.env` file.


In [ ]:
from typing import Any


@tool
def web_search(
    query: str,
    include_domains: list[str] | None = None,
    exclude_domains: list[str] | None = None,
    max_results: int = 5,
) -> dict[str, Any]:
    """Search the public web for current or external information using Tavily.

    Args:
        query: What information to search for.
        include_domains: Optional allowed domains, e.g. ["nist.gov", "oecd.org"].
        exclude_domains: Optional domains to exclude, e.g. ["reddit.com"].
        max_results: Number of results to return, clamped to 1-10.

    Use this tool when the question needs current external facts, industry research,
    public reports, or competitor information. Use domain filters when requested or
    when a domain restriction is useful. This tool does not query internal databases.
    """
    if not query or not query.strip():
        return {"error": "Please provide a non-empty search query."}
    if tavily_client is None:
        return {"error": "Tavily is not configured. Set TAVILY_API_KEY in the environment."}

    def normalize_domains(domains: list[str] | None) -> list[str] | None:
        if not domains:
            return None
        cleaned = []
        for domain in domains:
            if not domain or not domain.strip():
                continue
            value = domain.strip().lower()
            value = value.removeprefix("https://").removeprefix("http://").strip("/")
            if "/" in value:
                value = value.split("/", 1)[0]
            cleaned.append(value)
        return cleaned or None

    kwargs: dict[str, Any] = {
        "query": query.strip(),
        "max_results": max(1, min(int(max_results), 10)),
        "search_depth": "basic",
    }
    include = normalize_domains(include_domains)
    exclude = normalize_domains(exclude_domains)
    if include:
        kwargs["include_domains"] = include
    if exclude:
        kwargs["exclude_domains"] = exclude

    try:
        return tavily_client.search(**kwargs)
    except Exception as exc:
        return {"error": f"Tavily search failed: {type(exc).__name__}: {str(exc)[:300]}"}


In [ ]:
agent = create_agent(
    model=model,
    tools=[lookup_schema, lookup_examples, lookup_common, validate_query, run_query, web_search],
    system_prompt=SYSTEM_PROMPT,
)

## 10. Run a question and inspect the tool loop

The trace shows whether the agent first retrieves schema knowledge, then executes SQL, and finally answers based on the returned result.

In [ ]:
# Test retrieval tools independently before asking the agent a full question.
print(lookup_schema.invoke({"term": "product category identifier"}))
print("\n--- Similar SQL examples ---\n")
print(lookup_examples.invoke({"question": "top products by units sold"}))
print("\n--- Common knowledge ---\n")
print(lookup_common.invoke({"question": "how should revenue be calculated?"}))


## 11. Test schema lookup on its own

You can call the tool directly to see what semantic memory retrieves. This is only a retrieval test; it does not execute SQL.

In [ ]:
# Test retrieval tools independently before asking the agent a full question.
print(lookup_schema.invoke({"term": "product category identifier"}))
print("\n--- Similar SQL examples ---\n")
print(lookup_examples.invoke({"question": "top products by units sold"}))
print("\n--- Common knowledge ---\n")
print(lookup_common.invoke({"question": "how should revenue be calculated?"}))


## 12. Teaching narration: from tools to semantic memory

> “In Version 2, our agent had a brain and hands. It could query PostgreSQL, but it still needed reliable knowledge about table names, column names, joins, and business meanings.
>
> In Version 3, we add semantic memory through `lookup_schema`. Before writing SQL, the agent can search the knowledge base using the meaning of the question, not just an exact keyword match.
>
> The lookup tool retrieves schema knowledge. The query tool retrieves live data. One tells the agent **how the data is structured**; the other tells it **what the data contains**.”

| Tool | Purpose | Analogy |
|---|---|---|
| `lookup_schema(term)` | Retrieves schema knowledge by semantic similarity | **Memory / reference library** |
| `run_query(sql)` | Executes SQL against PostgreSQL | **Hands / action** |
| GPT-6 model | Chooses which tool to call and interprets results | **Brain / decision-maker** |

## 13. Important limitations

- Keep `kb_feed_ecom.xlsx` beside the notebook, or change `KB_PATH`.
- This example uses in-memory embeddings, so the index is rebuilt when the notebook starts.
- The workbook's examples may use a different SQL dialect; use schema and relationship knowledge, but ensure generated SQL matches PostgreSQL.
- The similarity threshold (`0.30`) and top-k (`8`) are starting values, not universal settings; evaluate them against representative questions.
- The database role should have read-only permissions. Basic SQL prefix checks are not a complete security validator.
- The model and embedding API must both be accessible through your API account.

**Next step:** Evaluate retrieval quality with a set of questions whose relevant tables and columns are known, then tune top-k and the similarity threshold.